# Agent Handoff Publisher

Turns one completed screening run into the **deterministic evidence envelope** the
Foundry report agent consumes. The model never touches Spark, never sees the pixel
table, and never invents a number: every quantitative claim it can make traces to an
`evidenceId` emitted here.

| | |
| --- | --- |
| **Reads** | `gold_rf1_band_summary`, `gold_rf1_risk_matrix`, `gold_rf1_risk_hotspots`, `silver_source_coverage`, `silver_source_features` |
| **Writes** | `Files/agent-handoff/<run-id>/report-input.json` + `_SUCCESS` |
| **Contract** | `agent-architecture/contracts/geohazard-report-input.schema.json` (schemaVersion 1.0) |

The notebook **fails closed**. If the gold tables disagree on totals, if a required
field is missing, or if any evidence reference is malformed, nothing is published and
the run is quarantined - which is the behaviour `architecture.md` requires before a
report is ever generated.

Run with `gold_lakehouse` attached as the default lakehouse.

## 1. Parameters

In [ ]:
PIPELINE_RUN_ID = ""
BRONZE_RADIUS_KM = 20.0
PIPELINE_NAME = "pl_bronze_ingestion"

## 2. Resolve the run

Same fallback as the gold notebook: an explicit pipeline run id wins, otherwise adopt
the most recent gold run so a standalone execution still describes a real run.

In [ ]:
import json
import re
import uuid
from datetime import datetime, timezone
from pathlib import Path

from pyspark.sql import functions as F

WS = notebookutils.runtime.context["currentWorkspaceId"]
ONELAKE_ENDPOINT = notebookutils.conf.get("trident.onelake.endpoint").replace("https://", "")
SILVER_LH_ID = notebookutils.lakehouse.get("silver_lakehouse", workspaceId=WS).id
SILVER_ABFSS = f"abfss://{WS}@{ONELAKE_ENDPOINT}/{SILVER_LH_ID}/Tables"

GENERATED_AT_UTC = datetime.now(timezone.utc).isoformat()

_raw_run_id = str(PIPELINE_RUN_ID or "").strip()
RUN_ID = re.sub(r"[^A-Za-z0-9._-]+", "-", _raw_run_id)[:128].strip(".-_") if _raw_run_id else ""

band_summary_all = spark.read.table("gold_rf1_band_summary")
if not RUN_ID or band_summary_all.filter(F.col("run_id") == RUN_ID).limit(1).count() == 0:
    _runs = sorted(row["run_id"] for row in band_summary_all.select("run_id").distinct().collect())
    if not _runs:
        raise RuntimeError("gold_rf1_band_summary is empty; run the gold notebook first.")
    print(f"  no gold rows for run_id={RUN_ID or '(blank)'}; adopting {_runs[-1]}")
    RUN_ID = _runs[-1]

RUN_FILTER = F.col("run_id") == RUN_ID
print(f"run_id: {RUN_ID}")


def read_silver_table(table_name):
    return spark.read.format("delta").load(f"{SILVER_ABFSS}/{table_name}")

## 3. Collect the deterministic evidence

Every value below is read from a gold or silver table. Nothing is computed here that
is not already published, so the report and the lakehouse can never disagree.

In [ ]:
EVIDENCE = []


def add_evidence(kind, title, source_table, values, query=None):
    """Register one citable fact and return its evidence id."""
    evidence_id = f"E{len(EVIDENCE) + 1}"
    record = {
        "id": evidence_id,
        "kind": kind,
        "title": title,
        "sourceTable": source_table,
        "retrievedAtUtc": GENERATED_AT_UTC,
        "values": values,
    }
    if query:
        record["query"] = query
    EVIDENCE.append(record)
    return evidence_id


BAND_RANGES = {"Low": (1, 4), "Moderate": (5, 9), "High": (10, 19), "Extreme": (20, 25)}
BAND_ORDER = ["Low", "Moderate", "High", "Extreme"]

# --- AOI and grid geometry, taken from the published pixel table ------------------
pixels = spark.read.table("gold_rf1_risk_pixels").filter(RUN_FILTER)
aoi_row = pixels.select("aoi_lat", "aoi_lon", "resolution_m", "aoi_name").first()
if aoi_row is None:
    raise RuntimeError(f"gold_rf1_risk_pixels has no rows for run_id={RUN_ID}.")

aoi_lat = float(aoi_row["aoi_lat"])
aoi_lon = float(aoi_row["aoi_lon"])
resolution_m = float(aoi_row["resolution_m"])
total_pixels = int(pixels.count())
pixel_area_km2 = (resolution_m ** 2) / 1_000_000.0
total_area_km2 = total_pixels * pixel_area_km2

utm_zone = min(60, max(1, int((aoi_lon + 180.0) // 6.0) + 1))
analysis_crs = f"EPSG:{(32600 if aoi_lat >= 0 else 32700) + utm_zone}"

# The analysis radius is half the grid width; recover it from the pixel extent.
extent = pixels.select(F.max("row").alias("max_row"), F.max("col").alias("max_col")).first()
analysis_radius_km = ((int(extent["max_col"]) + 1) * resolution_m / 2.0) / 1000.0

bronze_radius_km = float(BRONZE_RADIUS_KM)
try:
    features_all = read_silver_table("silver_source_features")
    radius_row = features_all.select("aoi_radius_km").first()
    if radius_row is not None and radius_row["aoi_radius_km"]:
        bronze_radius_km = float(radius_row["aoi_radius_km"])
except Exception as error:
    print(f"  (silver_source_features unavailable for bronze radius: {str(error).splitlines()[0][:90]})")

run_evidence_id = add_evidence(
    "run", "Pipeline run identity", "gold_rf1_risk_pixels",
    {"runId": RUN_ID, "pipelineName": PIPELINE_NAME, "totalPixels": total_pixels},
)
aoi_evidence_id = add_evidence(
    "aoi", "Area of interest and analysis grid", "gold_rf1_risk_pixels",
    {
        "latitude": aoi_lat, "longitude": aoi_lon,
        "bronzeRadiusKm": bronze_radius_km, "analysisRadiusKm": analysis_radius_km,
        "analysisCrs": analysis_crs, "resolutionM": resolution_m,
        "aoiName": aoi_row["aoi_name"],
    },
)
print(f"AOI {aoi_lat}, {aoi_lon} | {total_pixels:,} px @ {resolution_m:g} m "
      f"({analysis_crs}) | {total_area_km2:.4f} km2")

## 4. Source coverage

An empty source is a **data gap**, not a measured zero. The distinction is carried
explicitly so the report can never present missing coverage as an absence of hazard.

In [ ]:
source_coverage = []
try:
    coverage_df = read_silver_table("silver_source_coverage")
    if "run_id" in coverage_df.columns:
        coverage_runs = sorted(row["run_id"] for row in coverage_df.select("run_id").distinct().collect())
        chosen_run = RUN_ID if RUN_ID in coverage_runs else (coverage_runs[-1] if coverage_runs else None)
        if chosen_run and chosen_run != RUN_ID:
            print(f"  coverage recorded under {chosen_run}; using it for source lineage")
        if chosen_run:
            coverage_df = coverage_df.filter(F.col("run_id") == chosen_run)
    for row in coverage_df.orderBy("table_name").collect():
        record_count = None if row["record_count"] is None else int(row["record_count"])
        entry = {
            "sourceName": row["source_name"],
            "tableName": row["table_name"],
            "status": row["status"],
            "recordCount": record_count,
        }
        if row["note"]:
            entry["note"] = row["note"]
        entry["evidenceId"] = add_evidence(
            "sourceCoverage", f"Source coverage: {row['table_name']}", "silver_source_coverage",
            {"tableName": row["table_name"], "status": row["status"], "recordCount": record_count},
        )
        source_coverage.append(entry)
except Exception as error:
    print(f"  WARNING: silver_source_coverage unavailable ({str(error).splitlines()[0][:120]})")

if not source_coverage:
    # The contract requires at least one coverage entry; record the gap honestly.
    source_coverage.append({
        "evidenceId": add_evidence(
            "sourceCoverage", "Source coverage unavailable", "silver_source_coverage",
            {"status": "unavailable"},
        ),
        "sourceName": "Bronze source coverage",
        "tableName": "silver_source_coverage",
        "status": "unavailable",
        "recordCount": None,
        "note": "silver_source_features has not been run for this workspace",
    })

_status_counts = {}
for entry in source_coverage:
    _status_counts[entry["status"]] = _status_counts.get(entry["status"], 0) + 1
print(f"Source coverage: {_status_counts}")

## 5. Risk summary, matrix, and hotspots

In [ ]:
band_rows = {row["risk_band"]: row for row in
             spark.read.table("gold_rf1_band_summary").filter(RUN_FILTER).collect()}
bands = []
for band in BAND_ORDER:
    row = band_rows.get(band)
    if row is None:
        continue
    entry = {
        "band": band,
        "pixelCount": int(row["pixel_count"]),
        "areaKm2": float(row["area_km2"]),
        "pct": float(row["pct"]),
    }
    entry["evidenceId"] = add_evidence(
        "bandSummary", f"Risk band {band} extent", "gold_rf1_band_summary",
        {"band": band, "pixelCount": entry["pixelCount"],
         "areaKm2": entry["areaKm2"], "pct": entry["pct"]},
    )
    bands.append(entry)

risk_matrix = []
for row in (spark.read.table("gold_rf1_risk_matrix").filter(RUN_FILTER)
            .orderBy("c_rating", "s_rating").collect()):
    entry = {
        "sRating": int(row["s_rating"]),
        "cRating": int(row["c_rating"]),
        "riskScore": int(row["risk_score"]),
        "riskBand": row["risk_band"],
        "pixelCount": int(row["pixel_count"]),
    }
    entry["evidenceId"] = add_evidence(
        "riskMatrix", f"Matrix cell S={entry['sRating']} C={entry['cRating']}",
        "gold_rf1_risk_matrix", entry.copy(),
    )
    risk_matrix.append(entry)

hotspots = []
hotspot_context = []
try:
    hotspot_rows = (spark.read.table("gold_rf1_risk_hotspots").filter(RUN_FILTER)
                    .orderBy("rank").collect())
    for row in hotspot_rows:
        entry = {
            "featureId": row["hotspot_id"],
            "rank": int(row["rank"]),
            "latitude": float(row["centroid_lat"]),
            "longitude": float(row["centroid_lon"]),
            "sRating": int(row["s_rating"]),
            "cRating": int(row["c_rating"]),
            "riskScore": int(row["risk_score"]),
            "riskBand": row["risk_band"],
        }
        # The spatial "why" travels as evidence values rather than contract fields, so
        # the agent can explain a hotspot without the schema having to model geology.
        context = {
            "featureId": row["hotspot_id"],
            "areaKm2": float(row["area_km2"]),
            "pixelCount": int(row["pixel_count"]),
            "meanSlopeDeg": round(float(row["mean_slope_deg"]), 3),
            "meanElevationM": round(float(row["mean_elevation_m"]), 2),
            "soilName": row["soil_name"],
            "soilDrainageClass": row["soil_drainage_class"],
            "soilParentMaterial": row["soil_parent_material"],
            "bedrockUnit": row["bedrock_unit"],
        "bedrockAge": row["bedrock_age"],
            "landCover": row["worldcover_label"],
            "nearestFaultKm": (None if row["nearest_fault_km"] is None
                               else round(float(row["nearest_fault_km"]), 3)),
        }
        entry["evidenceId"] = add_evidence(
            "hotspot", f"Hotspot {row['hotspot_id']} (rank {entry['rank']})",
            "gold_rf1_risk_hotspots", {**entry, **context},
        )
        hotspots.append(entry)
        hotspot_context.append(context)
except Exception as error:
    print(f"  WARNING: gold_rf1_risk_hotspots unavailable ({str(error).splitlines()[0][:120]})")

print(f"bands={len(bands)}  matrixCells={len(risk_matrix)}  hotspots={len(hotspots)}")

## 6. Limitations

Derived from what the run actually saw, not boilerplate. Missing sources and thin soil
coverage become explicit caveats the agent is required to carry into the report.

In [ ]:
limitations = [
    "Screening-level output. RF-1 ratings use availability-weighted proxies over public "
    "data and are not a substitute for site investigation or engineering assessment.",
    "Susceptibility blends indirect remote-sensing proxies with surveyed soil polygons; "
    "only the soil survey is direct ground truth.",
]

_empty_sources = [entry["tableName"] for entry in source_coverage if entry["status"] == "empty"]
if _empty_sources:
    limitations.append(
        "The following configured sources returned no records over this AOI and must be "
        "treated as data gaps rather than measured zeros: " + ", ".join(sorted(_empty_sources)) + ".")

_unavailable = [entry["tableName"] for entry in source_coverage if entry["status"] in ("unavailable", "error")]
if _unavailable:
    limitations.append(
        "The following sources could not be read for this run: " + ", ".join(sorted(_unavailable)) + ".")

try:
    mapped_pixels = int(pixels.filter(F.col("soil_mapped") > 0).count())
    mapped_pct = mapped_pixels / total_pixels * 100.0 if total_pixels else 0.0
    limitations.append(
        f"Surveyed soil polygons cover {mapped_pct:.1f}% of the analysis grid "
        f"({mapped_pixels:,} of {total_pixels:,} pixels); elsewhere susceptibility rests on "
        "remote-sensing proxies alone.")
    add_evidence("limitation", "Soil survey coverage of the analysis grid", "gold_rf1_risk_pixels",
                 {"mappedPixels": mapped_pixels, "totalPixels": total_pixels,
                  "mappedPct": round(mapped_pct, 3)})
except Exception:
    pass

if not hotspots:
    limitations.append("No contiguous High or Extreme cluster met the minimum hotspot size, "
                       "so no ranked hotspots are available for this run.")

for limitation in limitations:
    print(f"  - {limitation}")

## 7. Assemble, validate, publish

Validation runs **before** anything is written, and `_SUCCESS` is written last. A
consumer that sees the completion marker is guaranteed a complete, self-consistent
document.

In [ ]:
webmap_manifest_uri = f"Files/gold_rf1_webmap/runs/{RUN_ID}/gold_rf1_webmap_manifest.json"
_manifest_path = Path("/lakehouse/default") / webmap_manifest_uri
if not _manifest_path.exists():
    print(f"  WARNING: web-map manifest not found at {webmap_manifest_uri}; "
          "the report will render with a map-unavailable state.")
    webmap_manifest_uri = None

report_input = {
    "schemaVersion": "1.0",
    "run": {
        "runId": RUN_ID,
        "pipelineName": PIPELINE_NAME,
        "status": "Completed",
        "completedAtUtc": GENERATED_AT_UTC,
    },
    "aoi": {
        "latitude": aoi_lat,
        "longitude": aoi_lon,
        "bronzeRadiusKm": bronze_radius_km,
        "analysisRadiusKm": analysis_radius_km,
        "analysisCrs": analysis_crs,
        "resolutionM": resolution_m,
    },
    "method": {
        "riskFactor": "RF-1",
        "formula": "risk_score = s_rating * c_rating",
        "bandDefinitions": [
            {"band": band, "minimumScore": low, "maximumScore": high}
            for band, (low, high) in BAND_RANGES.items()
        ],
    },
    "sourceCoverage": source_coverage,
    "riskSummary": {
        "totalPixels": total_pixels,
        "totalAreaKm2": total_area_km2,
        "bands": bands,
    },
    "riskMatrix": risk_matrix,
    "hotspots": hotspots,
    "limitations": limitations,
    "evidence": EVIDENCE,
}
if webmap_manifest_uri:
    report_input["webmapManifestUri"] = webmap_manifest_uri


def validate(document):
    """Fail-closed checks mirroring geohazard-report-input.schema.json."""
    problems = []

    for key in ("schemaVersion", "run", "aoi", "method", "sourceCoverage",
                "riskSummary", "riskMatrix", "hotspots", "limitations", "evidence"):
        if key not in document:
            problems.append(f"missing required key: {key}")
    if problems:
        return problems

    if document["schemaVersion"] != "1.0":
        problems.append("schemaVersion must be '1.0'")
    if document["run"]["status"] != "Completed":
        problems.append("run.status must be 'Completed'")
    if not 1 <= len(document["run"]["runId"]) <= 128:
        problems.append("run.runId must be 1-128 characters")

    aoi = document["aoi"]
    if not -80 <= aoi["latitude"] <= 84:
        problems.append("aoi.latitude out of range")
    if not -180 <= aoi["longitude"] <= 180:
        problems.append("aoi.longitude out of range")
    if not 0 < aoi["bronzeRadiusKm"] <= 100:
        problems.append("aoi.bronzeRadiusKm out of range")
    if not 0 < aoi["analysisRadiusKm"] <= 5:
        problems.append(f"aoi.analysisRadiusKm out of range: {aoi['analysisRadiusKm']}")
    if not re.match(r"^EPSG:(326|327)[0-9]{2}$", aoi["analysisCrs"]):
        problems.append(f"aoi.analysisCrs malformed: {aoi['analysisCrs']}")

    if len(document["method"]["bandDefinitions"]) != 4:
        problems.append("method.bandDefinitions must contain exactly 4 bands")
    if not document["sourceCoverage"]:
        problems.append("sourceCoverage must contain at least one entry")
    for entry in document["sourceCoverage"]:
        if entry["status"] not in ("available", "empty", "error", "unavailable"):
            problems.append(f"invalid sourceCoverage status: {entry['status']}")

    summary = document["riskSummary"]
    if summary["totalPixels"] < 1:
        problems.append("riskSummary.totalPixels must be >= 1")
    if not 1 <= len(summary["bands"]) <= 4:
        problems.append("riskSummary.bands must contain 1-4 entries")

    # Totals must reconcile across gold tables, or the run is quarantined.
    band_total = sum(entry["pixelCount"] for entry in summary["bands"])
    if band_total != summary["totalPixels"]:
        problems.append(f"band pixel total {band_total} != totalPixels {summary['totalPixels']}")
    matrix_total = sum(cell["pixelCount"] for cell in document["riskMatrix"])
    if matrix_total != summary["totalPixels"]:
        problems.append(f"matrix pixel total {matrix_total} != totalPixels {summary['totalPixels']}")

    if not 1 <= len(document["riskMatrix"]) <= 25:
        problems.append("riskMatrix must contain 1-25 cells")
    for cell in document["riskMatrix"]:
        if cell["sRating"] * cell["cRating"] != cell["riskScore"]:
            problems.append(f"matrix cell {cell['sRating']}x{cell['cRating']} riskScore mismatch")

    if len(document["hotspots"]) > 50:
        problems.append("hotspots exceeds the 50-feature cap")
    seen_features = set()
    for hotspot in document["hotspots"]:
        if hotspot["featureId"] in seen_features:
            problems.append(f"duplicate hotspot featureId: {hotspot['featureId']}")
        seen_features.add(hotspot["featureId"])
        if hotspot["sRating"] * hotspot["cRating"] != hotspot["riskScore"]:
            problems.append(f"hotspot {hotspot['featureId']} riskScore mismatch")

    if not document["limitations"]:
        problems.append("limitations must contain at least one entry")

    # Every evidence id must be well formed, unique, and actually referenced.
    declared = set()
    for record in document["evidence"]:
        if not re.match(r"^E[1-9][0-9]*$", record["id"]):
            problems.append(f"malformed evidence id: {record['id']}")
        if record["id"] in declared:
            problems.append(f"duplicate evidence id: {record['id']}")
        declared.add(record["id"])
        for key in ("kind", "title", "sourceTable", "retrievedAtUtc", "values"):
            if key not in record:
                problems.append(f"evidence {record['id']} missing {key}")

    referenced = set()
    for collection in (document["sourceCoverage"], summary["bands"],
                       document["riskMatrix"], document["hotspots"]):
        for entry in collection:
            referenced.add(entry["evidenceId"])
    unknown = referenced - declared
    if unknown:
        problems.append(f"references to undeclared evidence ids: {sorted(unknown)}")

    return problems


problems = validate(report_input)
if problems:
    for problem in problems:
        print(f"  FAIL: {problem}")
    raise RuntimeError(
        f"Report input failed contract validation ({len(problems)} problem(s)); "
        "run quarantined and nothing was published.")

OUTPUT_DIR = Path("/lakehouse/default/Files/agent-handoff") / RUN_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_INPUT_PATH = OUTPUT_DIR / "report-input.json"
HOTSPOT_CONTEXT_PATH = OUTPUT_DIR / "hotspot-context.json"
SUCCESS_PATH = OUTPUT_DIR / "_SUCCESS"

with REPORT_INPUT_PATH.open("w", encoding="utf-8") as stream:
    json.dump(report_input, stream, ensure_ascii=True, indent=2, default=str)
with HOTSPOT_CONTEXT_PATH.open("w", encoding="utf-8") as stream:
    json.dump({"runId": RUN_ID, "hotspots": hotspot_context}, stream,
              ensure_ascii=True, indent=2, default=str)

# Re-read what was persisted before declaring success.
with REPORT_INPUT_PATH.open("r", encoding="utf-8") as stream:
    persisted = json.load(stream)
if validate(persisted):
    raise RuntimeError("Persisted report input failed re-validation; run quarantined.")

SUCCESS_PATH.write_text(json.dumps({
    "schemaVersion": "1.0",
    "runId": RUN_ID,
    "generatedAtUtc": GENERATED_AT_UTC,
    "evidenceCount": len(EVIDENCE),
    "hotspotCount": len(hotspots),
    "artifacts": {
        "reportInput": f"Files/agent-handoff/{RUN_ID}/{REPORT_INPUT_PATH.name}",
        "hotspotContext": f"Files/agent-handoff/{RUN_ID}/{HOTSPOT_CONTEXT_PATH.name}",
    },
    "webmapManifestUri": webmap_manifest_uri,
}, ensure_ascii=True, indent=2), encoding="utf-8")

print(f"\nPublished agent handoff for run {RUN_ID}")
print(f"  {REPORT_INPUT_PATH}  ({REPORT_INPUT_PATH.stat().st_size:,} bytes)")
print(f"  evidence records : {len(EVIDENCE)}")
print(f"  hotspots         : {len(hotspots)}")
print(f"  contract         : schemaVersion 1.0, validated before and after write")

## 8. Preview the envelope

What the report agent receives. Note that no geometry, no pixel rows, and no raw
bronze property JSON crosses this boundary.

In [ ]:
preview = {key: value for key, value in report_input.items() if key != "evidence"}
preview["evidence"] = f"<{len(EVIDENCE)} evidence records>"
print(json.dumps(preview, indent=2, default=str)[:4000])